In [6]:
from typing import TypedDict, List, Dict
import random
from langgraph.graph import StateGraph, START, END

In [ ]:
class StateAgent(TypedDict):
    name: str
    target: int
    attempts: int
    guess: List[int]
    low: int
    high: int
    log: List[str]

In [102]:
def setup(state: StateAgent) -> StateAgent:
    """Initialize"""
    print("SETUP CALLED")
    state['name'] = f"Well Hello there {state['name']}"
    if 'log' not in state:
        state['log'] = []
    return state

def guess_node(state: StateAgent) -> StateAgent:
    """Random guess inside the current bounds"""
    if 'log' not in state:
        state['log'] = []
    new_guess = random.randint(state["low"], state["high"])
    state["guess"].append(new_guess)
    state["attempts"] += 1
    state["log"].append(f"Attempt {state['attempts']}: Guessed {new_guess}")
    return state

def hint_node(state: StateAgent) -> StateAgent:
    """Shrink the search window around the target"""
    if 'log' not in state:
        state['log'] = []
    latest_guess = state["guess"][-1]
    if latest_guess < state["target"]:
        state["low"] = latest_guess + 1     # too low -> raise floor
        state["log"].append(f"Hint: {latest_guess} is too low. New range: {state['low']}-{state['high']}")
    elif latest_guess > state["target"]:
        state["high"] = latest_guess - 1    # too high -> lower ceiling
        state["log"].append(f"Hint: {latest_guess} is too high. New range: {state['low']}-{state['high']}")
    return state

def should_continue(state: StateAgent) -> str:
    if 'log' not in state:
        state['log'] = []
    latest_guess = state["guess"][-1]
    if latest_guess == state["target"]:
        state["log"].append(f"SUCCESS: Found target {state['target']}!")
        return "correct"
    if state["attempts"] >= 7:
        state["log"].append("GAVE UP: Max attempts (7) reached!")
        return "give_up"
    state["log"].append("Looping for another attempt...")
    return "loop"

In [103]:
graph = StateGraph(StateAgent)

graph.add_node("guess", guess_node)
graph.add_node("hint", hint_node)
graph.add_node("intro", setup)

graph.add_edge(START, "intro")
graph.add_edge("intro", "guess")
graph.add_edge("guess", "hint")

graph.add_conditional_edges(
    "hint", 
    should_continue,
    {"loop": "guess", "correct": END, "give_up": END})

app = graph.compile()

In [109]:
result = app.invoke({"name": "Zam", "target": 4, "attempts": 0, "low": 1, "high": 20, "guess": [], "log": []})

# Format output as string
output = f"""
╔════════════════════════════════════════════════════════════════╗
║                     GUESSING GAME OUTPUT                       ║
╚════════════════════════════════════════════════════════════════╝

Player: {result['name']}
Target: {result['target']}

Execution Log:
--------------"""

# Recreate execution story
current_low = 1
current_high = 20

for i, guess in enumerate(result['guess'], 1):
    output += f"\nAttempt {i}: Guessed {guess}"
    
    if guess == result['target']:
        output += f" → SUCCESS! Found target {result['target']}!"
        break
    elif guess < result['target']:
        output += f" → Too low. New range: {guess + 1}-{current_high}"
        current_low = guess + 1
        if i < len(result['guess']):
            output += " (Looping for another attempt...)"
    elif guess > result['target']:
        output += f" → Too high. New range: {current_low}-{guess - 1}"
        current_high = guess - 1
        if i < len(result['guess']):
            output += " (Looping for another attempt...)"

# Summary
output += f"""

╔════════════════════════════════════════════════════════════════╗
║                           SUMMARY                              ║
╚════════════════════════════════════════════════════════════════╝
Total Attempts: {result['attempts']}
Guesses Made: {result['guess']}
"""

if result['guess'][-1] == result['target']:
    output += "Result: ✓ SUCCESS\n"
elif result['attempts'] >= 7:
    output += "Result: ✗ GAVE UP (max attempts reached)\n"
else:
    output += "Result: ONGOING\n"

print(output)

SETUP CALLED

╔════════════════════════════════════════════════════════════════╗
║                     GUESSING GAME OUTPUT                       ║
╚════════════════════════════════════════════════════════════════╝

Player: Well Hello there Zam
Target: 4

Execution Log:
--------------
Attempt 1: Guessed 14 → Too high. New range: 1-13 (Looping for another attempt...)
Attempt 2: Guessed 11 → Too high. New range: 1-10 (Looping for another attempt...)
Attempt 3: Guessed 2 → Too low. New range: 3-10 (Looping for another attempt...)
Attempt 4: Guessed 4 → SUCCESS! Found target 4!

╔════════════════════════════════════════════════════════════════╗
║                           SUMMARY                              ║
╚════════════════════════════════════════════════════════════════╝
Total Attempts: 4
Guesses Made: [14, 11, 2, 4]
Result: ✓ SUCCESS

